# The CARLA campaign

What the synthetic map established, demonstrated in real towns. Per the
2026-08-10 split, **CARLA's job is to show the result carries over, not to
re-establish it** — the 5% CI bar lives on the synthetic map, where a trial
costs seconds instead of minutes. So this runs enough seeds to demonstrate
transfer, and no more.

It covers **Town01 and every other town listed**, in one place. Town01 is not
special here: it is the town the calibration recipe was invented on, so it is
the reference the others are read against, but it runs the same scenario
under the same harness as the rest.

## Why the earlier CARLA runs were purged

Every CARLA run before 2026-08-11 was made against a travel model that
estimated `length / 30 km/h` for the whole town, which measured **1.75x
optimistic** — the policy planned pickup windows in a time base the bus
could not live in. Two things were wrong, and both are now fixed:

- **The speed limits were readable all along.** Town01 publishes 22
  type-274 landmarks at 30/60/90, and a driving vehicle reads them back.
  Its arterials — 37.7% of the town's length — were priced at 30.
- **What a bus achieves is not what the sign says.** 32% of every run is
  spent stationary at a red light. That is a bias, not noise (execution
  noise measures **0.034**), so it belongs in the estimate:
  `traffic_efficiency`, calibrated per town by `src/map/carla_calibrate.py`.

After both, the model predicts Town01's legs with **bias 1.000x** and
residual sigma **0.278**, down from 1.75x and 0.408. **These results are not
comparable with anything recorded before that date.**

## The four questions

1. **§3.2.2** — fulfilment >= 80%, and a positive sum of percentage
   differences against a fixed-route bus. The headline claim, per town.
2. **Does the ranking transfer between towns?** The 25-map synthetic sweep
   found the map moves the absolute numbers a great deal while leaving the
   *ranking* alone — `Reordering W750` won 22 of 25 and was never worse than
   2nd. If CARLA agrees, the recommendation is general rather than a fact
   about one town.
3. **Does L1's ride advantage survive real driving?** Synthetic: −111.5 s
   pooled over 25 maps, and the execution-noise study predicted it *widens*
   when driving is unreliable.
4. **Does the calibration hold, and is the window kept?** `traffic_efficiency`
   was measured on a fixed tour of every stop; a campaign drives whatever the
   policy picks. And the breach rate only became a real measurement once the
   clock stopped being 2.2x optimistic.

## The scenario is held identical across towns, deliberately

Same requests, same interval, same capacity, same window. Towns differ in
size, so an identical scenario is easier in a small town and harder in a
large one — **that variation is the finding**, not a confound to normalise
away. It is what the synthetic map-seed sweep did, and normalising per town
would destroy the comparison this exists to make. Only `MAX_TIME` scales,
generously: a ceiling that truncates a run looks exactly like a coverage
failure.

## Every town needs three artifacts first

| artifact | how | automated? |
|---|---|---|
| `<Town>.topology.json` | `src.map.carla_dump` | yes — section 2 |
| `<Town>.stops.json` | `notebooks/carla-stop-authoring.ipynb` | **no — hand-authored** |
| `traffic_efficiency` | `src.map.carla_calibrate` | yes — section 3 |

**`traffic_efficiency` does not transfer between towns.** It is dominated by
red-light density, a property of the map. Town01's 0.455 means nothing
anywhere else, and reusing it would repeat the error the old runs were
purged for.

## Running it

Needs a live server: `docker compose up -d`, and give it a minute. It is
**sequential** — `CarlaBackend` sets `parallel_safe = False`, one server
cannot host a campaign per core. Everything lands in the results store as it
completes, so **stopping and re-running continues rather than restarting**.

## Placeholders — all resolved 2026-08-11

Nothing here is a guess any more, except the per-trial ETAs, which only
feed the progress bar and self-correct from its rolling mean.

| what | resolved by |
|---|---|
| **Which towns** | Section 1's survey. Town07 was in the earlier list and **is not installed** — the image ships Town01–05 and Town10HD. |
| **Per-town calibration** | Section 3, run for all four towns. `traffic_efficiency` spans **0.328–0.497**, so it decisively does not transfer between towns. |
| **Minutes per trial** | Town01's 4.6 is measured over 30 trials; the rest are scaled estimates. |

!!! warning "Town03 and Town05's calibrations are survivorship-biased"
    8 and 12 of 30 calibration legs hit the 900 s `LEG_TIMEOUT_S` cap, and
    the legs that time out are the **long** ones — so both factors are
    fitted to the short legs that finished. At 3.18 m/s that cap is 2.9 km
    against Town05's 25.7 km of road. Raise the cap and re-calibrate before
    quoting either figure, and treat "is a 15-stop scenario even sensible on
    a 25 km map" as an open question.

## Configuration

In [ ]:
# Town01 is prepared and calibrated; it runs the same scenario as the rest.
REFERENCE_TOWN = "Town01"

# Settled by the section 1 survey, 2026-08-11. The server ships **Town01-05
# and Town10HD only** -- Town06 and Town07 need the separately downloaded
# Additional Maps, and Town07 was in the earlier placeholder, so it would
# have failed at runtime.
#
# Excluded: Town10HD publishes no speed-limit landmarks at all, so the
# extraction finds nothing, and it was already measured as barely able to
# move a bus; Town04's 37.9 km of road makes every trial expensive.
EXTRA_TOWNS = ["Town02", "Town03", "Town05"]

ALL_TOWNS = [REFERENCE_TOWN] + EXTRA_TOWNS

# ETA only. Town01's 4.6 is measured over 30 trials; the rest are scaled by
# road length and effective speed and are therefore guesses -- the bar's
# rolling mean will correct them within five trials.
EXPECTED_TRIAL_MINUTES = {"Town01": 4.6, "Town02": 4.5,
                          "Town03": 9.0, "Town05": 14.0}
DEFAULT_TRIAL_MINUTES = 6.0

# Measured 2026-08-11 by src.map.carla_calibrate, one tour-based calibration
# per town. traffic_efficiency runs 0.328-0.497 across these four, a 1.5x
# range, so it decisively does not transfer -- red-light share drives it.
#
# ⚠ STALE as of 2026-08-16 -- these were fitted under the turn-list steering
# that the project no longer uses, and under a 900 s cap that discarded 8 of
# 30 Town03 legs and 12 of 30 on Town05 (the long ones, so both were fitted
# to the short legs that finished). Direct drive is 55% faster and reaches
# stops the turn list never did. Section 3 re-fits all four; these values are
# kept only so the notebook runs before that happens.
CALIBRATION_REFERENCE = {
    "Town01": {"speed_mps": 4.21, "red_fraction": 0.321},   # efficiency 0.455
    "Town02": {"speed_mps": 3.36, "red_fraction": 0.324},   # efficiency 0.410
    "Town03": {"speed_mps": 4.27, "red_fraction": 0.370},   # efficiency 0.497
    "Town05": {"speed_mps": 3.18, "red_fraction": 0.552},   # efficiency 0.328
}

# --- The scenario, identical in every town --------------------------------
#
# Calibrated 2026-08-11 by sweeping demand and patience on Town01's real
# geometry against the MOCK backend -- seconds a trial instead of minutes,
# no server. The first attempt (12 requests / 150 s / patience 1800) had
# *fixed route* serving 100%, which makes the fulfilment gate inert and
# collapses §3.2.2 onto the sum alone. Raising the arrival rate did not fix
# it: at one request per 60 s everything still served ~100%.
#
# **Patience is the lever, not the arrival rate.** At 1800 s nobody ever
# gives up, so the bus always catches up eventually. The sweep:
#
#   patience   1800    1200     900     750     600
#   fixed     100.0%  100.0%   77.9%   62.5%   52.5%
#
NPC_VEHICLES = 20         # traffic costs ~3x wall clock and changes service
                          # not at all, but it is what the noise study models
SEEDS = list(range(10))   # demonstration of transfer, not a 5% CI
MAX_REQUESTS = 30
DEMAND_INTERVAL_S = 90.0
CAPACITY = 20

# window == patience is the recommendation being demonstrated: coverage peaks
# where the promise matches what the passenger will tolerate. Measured here:
# fixed route 62.3%, both contenders 93.3%, so the gate does real work -- and
# L1 - L0 mean ride comes out at -70 s, which is exactly the synthetic campus
# map's figure over 60 seeds.
WINDOW = 750.0
PATIENCE = 750.0

DWELL_S = 10.0
FIXED_DELTA_S = 0.05      # never above 0.1; past CARLA's substepping limit

# Periodic re-issue of the turn list. **Dead under direct drive** -- it
# existed because set_route hands over one list per leg and never refreshes
# it, so a list that ran out left the bus navigating at random. CARLA's own
# agent follows the planned waypoints and cannot take a turn we did not plan,
# so there is nothing to re-establish; CarlaBackend ignores this outright.
REPLAN_EVERY_S = None
                          # the physics silently stops keeping up
MAX_TIME = 40000.0        # generous: a ceiling that truncates a run looks
                          # exactly like a coverage failure

CARLA_HOST, CARLA_PORT = "localhost", 2000

In [ ]:
import statistics
import time

import matplotlib.pyplot as plt
import pandas as pd
from cycler import cycler
from scipy import stats
from tqdm.auto import tqdm

from src.algorithm.FixedRoutePolicy import FixedRoutePolicy
from src.algorithm.ReorderingInsertionPolicy import ReorderingInsertionPolicy
from src.algorithm.WindowInsertionPolicy import WindowInsertionPolicy
from src.experiment import run_campaign
from src.experiment.store import DEFAULT_RUNS_DIR, load_runs, to_results
from src.experiment.summary import summarize_results
from src.map.carla_stops import stops_path
from src.map.carla_topology import topology_path
from src.map.CarlaMapProvider import CarlaMapProvider
from src.simulation.CarlaBackend import CarlaBackend, stop_positions_from

from IPython.display import Markdown, display

# Validated palette slots 1-3 (all-pairs CVD 9.2, normal-vision 27.6 on this
# surface). Every chart ships with its table, which is the relief the green
# slot's sub-3:1 contrast requires.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a"]
SURFACE, INK, INK_SOFT, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e1"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "axes.prop_cycle": cycler(color=SERIES),
    "axes.edgecolor": INK_SOFT, "axes.labelcolor": INK_SOFT,
    "axes.titlecolor": INK, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.labelsize": 9, "axes.spines.top": False, "axes.spines.right": False,
    "grid.color": GRID, "grid.linewidth": 0.8, "xtick.color": INK_SOFT,
    "ytick.color": INK_SOFT, "xtick.labelsize": 9, "ytick.labelsize": 9,
    "legend.frameon": False, "legend.fontsize": 9, "figure.dpi": 110,
})

POLICY_LABELS = ["Fixed route", "Window W750", "Reordering W750"]
POLICIES = [
    ("Fixed route", lambda g: FixedRoutePolicy(g)),
    ("Window W750", lambda g: WindowInsertionPolicy(g, max_wait_s=WINDOW)),
    ("Reordering W750",
     lambda g: ReorderingInsertionPolicy(g, max_wait_s=WINDOW)),
]

TRIAL = dict(
    max_requests=MAX_REQUESTS, demand_interval_s=DEMAND_INTERVAL_S,
    capacity=CAPACITY, max_wait_s=PATIENCE, dwell_s=DWELL_S,
    max_time=MAX_TIME,
)

# A breach is a *fulfilled* rider whose wait exceeded the window. At
# window == patience that is impossible by construction: anyone who waits
# that long has already been cancelled, so abandonment enforces the promise
# on the policy's behalf. Section 10 reports this rather than plotting a
# flat zero and calling it a measurement.
BREACH_MEASURABLE = WINDOW < PATIENCE

print(f"{len(ALL_TOWNS)} towns x {len(POLICIES)} policies x {len(SEEDS)} "
      f"seeds = {len(ALL_TOWNS) * len(POLICIES) * len(SEEDS)} trials")
print(f"window {WINDOW:.0f}s, patience {PATIENCE:.0f}s -> breach rate is "
      + (f"observable in the {PATIENCE - WINDOW:.0f}s band between them"
         if BREACH_MEASURABLE else
         "STRUCTURALLY ZERO (window == patience, the operating point)"))

## 1. Survey the towns  🖥 *needs a live server*

Resolves the `EXTRA_TOWNS` placeholder with evidence. Loads each installed
town and measures what should drive the choice: how much road, how much of
it junction, how varied the speed limits — and, decisively, whether it has
**Shoulder** lanes, since every authored stop needs one to its right and
passengers are spawned there. A town without them cannot be used at all.

Skip this once the town list is settled.

In [ ]:
def survey_town(client, town: str) -> dict:
    """
    Measure one town's road network on a live server.
    :param client: Connected carla.Client.
    :param town: Town name to load and measure.
    :return: One row of survey metrics.
    """
    import carla

    world = client.load_world(town)
    carla_map = world.get_map()

    length = junction_length = 0.0
    for entry, exit_waypoint in carla_map.get_topology():
        span = entry.transform.location.distance(
            exit_waypoint.transform.location)
        length += span
        if entry.is_junction:
            junction_length += span

    waypoints = carla_map.generate_waypoints(5.0)
    driving = [w for w in waypoints if w.lane_type == carla.LaneType.Driving]
    with_shoulder = sum(
        1 for w in driving
        if (right := w.get_right_lane()) is not None
        and right.lane_type == carla.LaneType.Shoulder
    )
    landmarks = carla_map.get_all_landmarks_of_type("274")

    return {
        "town": town,
        "road_km": round(length / 1000, 2),
        "junction_%": round(100 * junction_length / length, 1) if length else 0,
        "driving_wps": len(driving),
        "shoulder_%": (round(100 * with_shoulder / len(driving), 1)
                       if driving else 0),
        "speed_signs": len(landmarks),
        "speed_values": sorted({float(l.value) for l in landmarks}),
        "traffic_lights": len(
            world.get_actors().filter("traffic.traffic_light*")),
    }


# Off by default: it reloads every installed town, and is only
# needed once, to settle EXTRA_TOWNS.
RUN_SURVEY = False   # set True to run; needs a live server

if RUN_SURVEY:
    import carla
    client = carla.Client(CARLA_HOST, CARLA_PORT)
    client.set_timeout(120.0)
    # The *_Opt variants are the same towns with toggleable layers; surveying
    # both doubles the work and tells us nothing new.
    candidates = [n.split("/")[-1] for n in
                  sorted(set(client.get_available_maps()))]
    candidates = [n for n in candidates if not n.endswith("_Opt")]
    print(f"server has {len(candidates)} towns: {candidates}")

    rows = []
    for town in tqdm(candidates, desc="surveying towns", unit="town"):
        try:
            rows.append(survey_town(client, town))
        except Exception as error:              # noqa: BLE001
            tqdm.write(f"  {town}: FAILED {error!r}")
    survey = pd.DataFrame(rows).set_index("town")
    display(survey)
    display(Markdown(
        "**Reading this.** `shoulder_%` near zero is disqualifying — no stop "
        "can be authored there without relaxing the legality rule. Otherwise "
        "prefer *variety* in `road_km`, `junction_%` and `speed_values` over "
        "picking the biggest; Town01 already covers small-grid-urban."))
else:
    print("survey skipped -- set RUN_SURVEY = True with a server up")

## 2. What is prepared, and what is missing  *(no server needed)*

Run any time. Stop authoring is manual and cannot be done from here.

In [ ]:
def readiness(town: str) -> dict:
    """
    Report which of a town's three artifacts exist.
    :param town: Town name.
    :return: A row describing what is present and what is not.
    """
    has_topology, has_stops = topology_path(town).exists(), stops_path(town).exists()
    efficiency, stops, note = None, None, ""
    if has_topology and has_stops:
        try:
            provider = CarlaMapProvider(town)
            efficiency, stops = provider.traffic_efficiency(), len(provider.get_stops())
            problems = provider.validate()
            note = "OK" if not problems else f"{len(problems)} map problems"
        except Exception as error:              # noqa: BLE001
            note = f"failed to load: {error!r}"
    return {
        "town": town,
        "topology": "yes" if has_topology else "MISSING",
        "stops": stops if stops is not None else ("yes" if has_stops else "MISSING"),
        "traffic_efficiency": efficiency if efficiency is not None else "—",
        "ready": "yes" if efficiency not in (None, 1.0) else "no",
        "note": note,
    }


display(pd.DataFrame([readiness(t) for t in ALL_TOWNS]).set_index("town"))

needs_stops = [t for t in ALL_TOWNS if not stops_path(t).exists()]
needs_calibration = [
    t for t in ALL_TOWNS
    if topology_path(t).exists() and stops_path(t).exists()
    and CarlaMapProvider(t).traffic_efficiency() == 1.0
]
READY_TOWNS = [t for t in ALL_TOWNS
               if t not in needs_stops and t not in needs_calibration]

if needs_stops:
    display(Markdown(
        f"> **Hand-authoring needed for: {', '.join(needs_stops)}.** Run "
        "`notebooks/carla-stop-authoring.ipynb` once per town. This is the "
        "one step nothing here can do for you."))
if needs_calibration:
    display(Markdown(
        f"> **Uncalibrated: {', '.join(needs_calibration)}.** Section 3 "
        "fixes this. A `traffic_efficiency` of 1.0 assumes the bus achieves "
        "the posted limit exactly — measured **1.75x optimistic** on Town01."))
print(f"ready to run: {READY_TOWNS}")

## 3. Prepare the towns  🖥 *needs a live server*

Dumps the road network and calibrates what a bus achieves on it. Skips towns
with no authored stops, since calibration drives between them, and skips
towns already calibrated — re-calibrating compounds, because the drive
happens under the factor already in force.

Budget 10–15 minutes per town: the dump is quick, the calibration drives
three full tours of every stop.

In [ ]:
CALIBRATION_TOURS = 3

RUN_PREPARE = True    # set False to skip; needs a live server

# traffic_efficiency is fitted BY DRIVING, so it belongs to the steering it
# was measured under. Everything on disk was fitted against the turn list;
# direct drive is 55% faster and reaches stops the turn list never did, so
# reusing those factors would have the policy plan against a bus it does not
# have -- and wrongly by a different amount per town, which is exactly what
# would poison the cross-town comparison in section 7.
#
# Set False once all four towns have been re-fitted under direct drive.
RECALIBRATE = True

if RUN_PREPARE:
    from src.map.carla_calibrate import calibrate
    from src.map.carla_dump import dump_world
    from src.map.carla_topology import (CarlaTopology, load_topology,
                                        save_topology)

    if RECALIBRATE:
        # Every prepared town, whether or not it already carries a factor.
        todo = [t for t in ALL_TOWNS if stops_path(t).exists()]
    else:
        todo = [t for t in ALL_TOWNS
                if stops_path(t).exists()
                and CarlaMapProvider(t).traffic_efficiency() == 1.0
                if topology_path(t).exists()]
        todo += [t for t in ALL_TOWNS
                 if stops_path(t).exists() and not topology_path(t).exists()]
    print(f"preparing {todo or 'nothing -- all prepared or awaiting stops'}")

    if todo:
        # Connect only when there is work; a server is not needed
        # merely to discover every town is already prepared.
        import carla
        client = carla.Client(CARLA_HOST, CARLA_PORT)
        client.set_timeout(120.0)

    for town in tqdm(todo, desc="preparing towns", unit="town"):
        steps = tqdm(total=2, desc=f"  {town}", leave=False, unit="step")

        steps.set_postfix_str("dumping roads")
        world = client.load_world(town)
        topology = dump_world(world)
        save_topology(topology, note=(
            f"dumped for the CARLA campaign; "
            f"{len(topology.speed_limit_overrides)} signed roads"))
        steps.update(1)

        steps.set_postfix_str(f"calibrating, {CALIBRATION_TOURS} tours")
        summary = calibrate(town, CALIBRATION_TOURS, NPC_VEHICLES,
                            CARLA_HOST, CARLA_PORT, verbose=False,
                            replan_every_s=REPLAN_EVERY_S,
                            direct_drive=True)
        save_topology(
            CarlaTopology(
                map=topology.map,
                default_speed_limit_kph=topology.default_speed_limit_kph,
                speed_limit_overrides=topology.speed_limit_overrides,
                segments=topology.segments,
                traffic_efficiency=summary["traffic_efficiency"]),
            note=(f"traffic_efficiency {summary['traffic_efficiency']} over "
                  f"{summary['legs_arrived']} legs at {NPC_VEHICLES} NPCs"))
        CALIBRATION_REFERENCE[town] = {
            "speed_mps": summary["effective_speed_mps"],
            "red_fraction": summary["red_light_fraction"]}
        steps.update(1)
        steps.close()
        tqdm.write(
            f"  {town}: efficiency {summary['traffic_efficiency']}, "
            f"bias {summary['bias']}x, sigma within "
            f"{summary['sigma_within_leg']} / between "
            f"{summary['sigma_between_legs']}, "
            f"red {100 * summary['red_light_fraction']:.0f}%")

    if todo:
        display(Markdown(
            "**Copy `CALIBRATION_REFERENCE` into the config cell** so "
            "section 8 still has it after a kernel restart:\n\n```python\n"
            f"CALIBRATION_REFERENCE = {CALIBRATION_REFERENCE}\n```"))
else:
    print("preparation skipped -- set RUN_PREPARE = True with a server up")

## 4. Run the campaign  🖥 *needs a live server*

Sequential, and per seed rather than per campaign so a CARLA hiccup costs one
trial instead of the night. Everything already recorded is skipped, so
re-running resumes.

Two nested bars — towns outside, seeds inside — with a **rolling mean of the
last five trials**, so the ETA reflects what the current town is costing
rather than what Town01 cost.

In [ ]:
def world_for(town: str, provider: CarlaMapProvider) -> dict:
    """
    Build the store identity for one town's world.

    ``traffic_efficiency`` and the speed-limit rule are in here because they
    change every estimate the policy makes; omitting them would let a
    pre-calibration run answer for a post-calibration one.
    :param town: Town name.
    :param provider: That town's loaded provider.
    :return: The world identity.
    """
    return {
        "town": town, "backend": "CarlaBackend",
        "npc_vehicles": NPC_VEHICLES, "fixed_delta_s": FIXED_DELTA_S,
        "no_rendering": True, "blueprint": "vehicle.volkswagen.t2_2021",
        "traffic_efficiency": provider.traffic_efficiency(),
        "speed_limits": "landmark-harmonic-v1",
        # How the bus was driven. Without this a run recorded under the old
        # turn-list steering has the SAME identity as one driven by CARLA's
        # agent, so run_campaign would skip the new trial as "already
        # recorded" and the collect cell would pool the two silently.
        "steering": "direct-drive-v1",
    }


def backend_factory_for(town: str, positions: dict):
    """
    Build a per-seed world builder for one town.
    :param town: Town to drive in.
    :param positions: That town's stop coordinates.
    :return: A function of seed returning a backend factory.
    """
    def for_seed(seed: int):
        def build(graph, start_stop_id: str) -> CarlaBackend:
            """Build the CARLA backend for one trial."""
            return CarlaBackend(
                stop_positions=positions, position=start_stop_id, town=town,
                host=CARLA_HOST, port=CARLA_PORT,
                fixed_delta_s=FIXED_DELTA_S, dwell_s=DWELL_S,
                npc_vehicles=NPC_VEHICLES, seed=seed, no_rendering=True,
                replan_every_s=REPLAN_EVERY_S,
                # Explicit even though it is the default, because the whole
                # campaign is only comparable if every trial drove the same
                # way -- see the "steering" key in world_for.
                direct_drive=True,
                # CARLA results are not independent of server history: the
                # last four-town campaign lost 7 Town05 runs to a
                # back-to-back-trials artefact and had to be discarded.
                reload_world=True)
        return build
    return for_seed


RUN_CAMPAIGN = True   # set False to skip; needs a live server

# Section 2 worked READY_TOWNS out *before* section 3 calibrated anything, so
# a town prepared in this same pass is still missing from it -- and the
# campaign would then silently skip the very town it just spent a quarter of
# an hour preparing. Recompute from what is on disk now.
READY_TOWNS = [
    t for t in ALL_TOWNS
    if stops_path(t).exists() and topology_path(t).exists()
    and CarlaMapProvider(t).traffic_efficiency() != 1.0
]
print(f"campaign will run: {READY_TOWNS}")
missing = [t for t in ALL_TOWNS if t not in READY_TOWNS]
if missing:
    display(Markdown(
        f"> **Not running: {', '.join(missing)}.** Still uncalibrated or "
        "missing an artifact — see sections 2–3."))

if RUN_CAMPAIGN:
    if not READY_TOWNS:
        display(Markdown(
            "> **Nothing to run.** No town is prepared — see sections 2–3."))

    failures, durations = [], []
    towns_bar = tqdm(READY_TOWNS, desc="towns", unit="town")
    for town in towns_bar:
        towns_bar.set_postfix_str(town)
        provider = CarlaMapProvider(town)
        positions = stop_positions_from(provider)
        world = world_for(town, provider)
        for_seed = backend_factory_for(town, positions)
        expected = EXPECTED_TRIAL_MINUTES.get(town, DEFAULT_TRIAL_MINUTES)

        # One tick per *trial*, not per seed. A seed is three policies and
        # roughly ten minutes, which is far too coarse to tell a slow run
        # from a stalled one.
        trials_bar = tqdm(total=len(SEEDS) * len(POLICIES), unit="trial",
                          desc=f"  {town}", leave=False)
        # Seed-major so the three policies of a seed finish together: an
        # interrupted run then leaves paired data rather than an orphan arm.
        for seed in SEEDS:
            for label, factory in POLICIES:
                started = time.perf_counter()
                ran = 0
                try:
                    result = run_campaign(
                        policies=[(label, factory)], seeds=[seed],
                        world=world, directory=DEFAULT_RUNS_DIR,
                        provider=provider, backend_factory_for=for_seed,
                        n_jobs=1, progress=False, **TRIAL)
                    ran = result.ran
                except Exception as error:      # noqa: BLE001
                    # Broad on purpose: a CARLA failure surfaces as almost
                    # anything, and losing the remaining trials is worse.
                    failures.append((town, seed, label, repr(error)))
                    tqdm.write(f"  {town} seed {seed} {label} FAILED: "
                               f"{error!r}")

                elapsed = (time.perf_counter() - started) / 60
                # A skipped trial costs no time; counting it would make the
                # ETA nonsense for the rest of a resumed run.
                if ran:
                    durations.append(elapsed)
                rolling = (statistics.fmean(durations[-5:]) if durations
                           else expected)
                trials_bar.update(1)
                trials_bar.set_postfix_str(
                    f"seed {seed} {label}"
                    + ("" if ran else " (already recorded)")
                    + f" | rolling {rolling:.1f} min/trial, "
                      f"expected {expected:.1f}")
        trials_bar.close()

    print(f"\ndone — {len(durations)} trials run, "
          f"{len(SEEDS) * len(POLICIES) * len(READY_TOWNS) - len(durations)} "
          f"skipped or failed, {len(failures)} failures")
    for town, seed, label, error in failures:
        print(f"  {town} seed {seed} {label}: {error}")
else:
    print("campaign skipped -- set RUN_CAMPAIGN = True with a server up")

## 5. Collect

In [ ]:
records = [
    r for r in load_runs(DEFAULT_RUNS_DIR,
                         where={"identity.world.backend": "CarlaBackend"})
    if r.identity.world.get("town") in ALL_TOWNS
    and r.identity.world.get("npc_vehicles") == NPC_VEHICLES
    and r.identity.world.get("speed_limits") == "landmark-harmonic-v1"
    # Without this the frame would silently pool runs driven by CARLA's
    # agent with any recorded under the old turn-list steering.
    and r.identity.world.get("steering") == "direct-drive-v1"
    and r.identity.trial.get("max_requests") == MAX_REQUESTS
    and r.identity.policy in POLICY_LABELS
]

frame = pd.DataFrame([{
    "town": r.identity.world["town"], "policy": r.identity.policy,
    "seed": r.identity.seed,
    "served": r.kpi.served_fraction * 100,
    "wait": r.kpi.mean_wait_time, "ride": r.kpi.mean_ride_time,
    "throughput": r.kpi.throughput,
    "breach": (100.0 * sum(w > WINDOW for w in r.wait_times)
               / len(r.wait_times) if r.wait_times else float("nan")),
    "sim_s": r.backend_stats.get("ticks", 0) * FIXED_DELTA_S,
    "distance_m": r.backend_stats.get("distance_m", 0.0),
    "red_s": r.backend_stats.get("red_light_ticks", 0) * FIXED_DELTA_S,
    # Under direct drive a repath means the bus was physically stuck --
    # the red-light and queueing exemptions cover the legitimate stops --
    # so this is effectively a collision counter. Measured at 1 per 56 legs.
    "repaths": r.backend_stats.get("repaths", 0),
    "unroutable": r.backend_stats.get("unroutable_legs", 0),
    "queued_s": r.backend_stats.get("queued_ticks", 0) * FIXED_DELTA_S,
} for r in records], columns=[
    "town", "policy", "seed", "served", "wait", "ride", "throughput",
    "breach", "sim_s", "distance_m", "red_s", "repaths", "unroutable",
    "queued_s"])

HAVE_RUNS = not frame.empty
TOWNS_WITH_RUNS = sorted(frame["town"].unique()) if HAVE_RUNS else []
print(f"{len(frame)} runs across {len(TOWNS_WITH_RUNS)} towns: "
      f"{TOWNS_WITH_RUNS}")

if HAVE_RUNS:
    display(frame.groupby(["town", "policy"])[
        ["served", "wait", "ride", "throughput"]].mean().round(1))
    suspect = frame[(frame["repaths"] > 0) | (frame["unroutable"] > 0)]
    print(f"runs with repaths or unroutable legs: {len(suspect)}"
          + ("" if suspect.empty else "  <- inspect before trusting these"))
else:
    display(Markdown(
        "> **No runs yet.** Run section 4, or check that the world identity "
        "here matches what it filed under — one differing field files runs "
        "under an identity this filter misses."))

## 6. §3.2.2 — the headline claim, per town

Fulfilment **>= 80%** and a **positive sum** of percentage differences
against fixed route across wait, ride and throughput.

The gate does the real work. Wait and ride are computed over *fulfilled*
requests only, so a policy that abandons its hard riders scores well on
them — the sum must never be read on its own.

In [ ]:
if not HAVE_RUNS:
    display(Markdown("_Nothing to report until the campaign has run._"))
else:
    for town in TOWNS_WITH_RUNS:
        block = [r for r in records if r.identity.world["town"] == town]
        display(Markdown(f"### {town}"))
        display(Markdown(summarize_results(
            to_results(block, paired=False)).markdown))

In [ ]:
if not HAVE_RUNS:
    display(Markdown("_Nothing to report until the campaign has run._"))
else:
    lines = ["| town | policy | fulfilment | wait Δ% | ride Δ% "
             "| throughput Δ% | **sum** | gate | §3.2.2 |",
             "|---|---|---|---|---|---|---|---|---|"]
    for town in TOWNS_WITH_RUNS:
        means = frame[frame["town"] == town].groupby("policy")[
            ["served", "wait", "ride", "throughput"]].mean()
        if "Fixed route" not in means.index:
            lines.append(f"| {town} | — | _no fixed-route baseline_ "
                         "| | | | | | |")
            continue
        base = means.loc["Fixed route"]
        for label in POLICY_LABELS[1:]:
            if label not in means.index:
                continue
            row = means.loc[label]
            # Lower is better for wait and ride, higher for throughput.
            d_wait = 100 * (base["wait"] - row["wait"]) / base["wait"]
            d_ride = 100 * (base["ride"] - row["ride"]) / base["ride"]
            d_tput = (100 * (row["throughput"] - base["throughput"])
                      / base["throughput"])
            total = d_wait + d_ride + d_tput
            gate = row["served"] >= 80.0
            lines.append(
                f"| {town} | {label} | {row['served']:.1f}% | {d_wait:+.1f} "
                f"| {d_ride:+.1f} | {d_tput:+.1f} | **{total:+.1f}** "
                f"| {'✓' if gate else '✗ <80%'} "
                f"| {'**PASS**' if gate and total > 0 else 'FAIL'} |")
        lines.append(
            f"| {town} | _fixed route_ | {base['served']:.1f}% | baseline "
            f"| {base['wait']:.0f} s | {base['ride']:.0f} s | | | |")
    display(Markdown("\n".join(lines)))

## 7. Does the ranking transfer between towns?

The synthetic sweep found the map moves absolute numbers a great deal while
leaving the *ranking* alone. Flat lines on the right mean the recommendation
is general; crossing lines mean it is a fact about particular towns.

In [ ]:
if not HAVE_RUNS or len(TOWNS_WITH_RUNS) < 2:
    display(Markdown(
        "_Needs at least two towns with runs; showing coverage only._"
        if HAVE_RUNS else
        "_Nothing to report until the campaign has run._"))
    if HAVE_RUNS:
        display(frame.groupby(["town", "policy"])["served"].mean()
                .unstack().round(1))
else:
    means = frame.groupby(["town", "policy"])["served"].mean().unstack()
    ranked = means.rank(axis=1, ascending=False).astype(int)
    towns = list(means.index)

    figure, axes = plt.subplots(1, 2, figsize=(12.5, 4.0))
    width = 0.26
    for offset, (colour, label) in enumerate(zip(SERIES, POLICY_LABELS)):
        if label in means:
            axes[0].bar([i + (offset - 1) * width for i in range(len(towns))],
                        means[label], width * 0.92, color=colour, label=label)
    axes[0].axhline(80, ls="--", lw=1.1, color=INK_SOFT)
    axes[0].set_xticks(range(len(towns)), towns)
    axes[0].set_ylabel("served (%)")
    axes[0].set_title("Coverage per town (dashed = the 80% gate)")
    axes[0].grid(axis="y"); axes[0].set_axisbelow(True); axes[0].legend()

    for colour, label in zip(SERIES, POLICY_LABELS):
        if label in ranked:
            axes[1].plot(range(len(towns)), ranked[label], marker="o",
                         markersize=8, lw=2, color=colour, label=label)
    axes[1].set_xticks(range(len(towns)), towns)
    axes[1].set_yticks([1, 2, 3]); axes[1].invert_yaxis()
    axes[1].set_ylabel("rank by coverage")
    axes[1].set_title("Ranking — flat lines mean the ranking transfers")
    axes[1].grid(axis="y"); axes[1].set_axisbelow(True); axes[1].legend()
    figure.tight_layout()
    plt.show()

    display(means.round(1))
    display(Markdown(
        "**Won by:** " + ", ".join(
            f"{t} — {w}" for t, w in means.idxmax(axis=1).items())
        + "\n\nSynthetic reference: `Reordering W750` won **22 of 25** maps "
        "and was never worse than 2nd; only the top two ever swapped."))

## 8. Does L1's advantage survive real driving?

Paired by seed, since both policies met the same demand in the same town.
Synthetic reference: ride **−111.5 s** pooled over 25 maps (p≈2e-138),
**−70 s** on the campus map alone.

In [ ]:
if not HAVE_RUNS:
    display(Markdown("_Nothing to report until the campaign has run._"))
else:
    lines = ["| town | metric | Window W750 | Reordering W750 | L1 − L0 "
             "| p | seeds |", "|---|---|---|---|---|---|---|"]
    for town in TOWNS_WITH_RUNS:
        block = frame[frame["town"] == town]
        l0 = block[block["policy"] == "Window W750"].set_index("seed")
        l1 = block[block["policy"] == "Reordering W750"].set_index("seed")
        shared = sorted(set(l0.index) & set(l1.index))
        if len(shared) < 2:
            continue
        for column, unit in [("served", "%"), ("wait", " s"), ("ride", " s")]:
            a = l0.loc[shared, column].astype(float)
            b = l1.loc[shared, column].astype(float)
            p = (stats.ttest_rel(b, a).pvalue if (b - a).std() > 0
                 else float("nan"))
            lines.append(
                f"| {town} | mean {column} | {a.mean():.1f}{unit} "
                f"| {b.mean():.1f}{unit} | **{(b - a).mean():+.1f}{unit}** "
                f"| {p:.4f} | {len(shared)} |")
    display(Markdown("\n".join(lines)))
    display(Markdown(
        "At ten seeds a town, treat the **sign and rough magnitude** as the "
        "claim rather than the p-value. A null here is weak evidence either "
        "way — the statistical case lives on the synthetic map."))

## 9. Did the calibration hold?

`traffic_efficiency` was measured on a fixed tour of every stop. A campaign
drives whatever the policy chooses, which is a different sample of the town
— so this checks the factor transfers off its own tour. Within a few percent
means it describes the town, and that adding a map is a mechanical procedure
rather than a research question.

In [ ]:
if not HAVE_RUNS:
    display(Markdown("_Nothing to report until the campaign has run._"))
else:
    drive = frame[frame["sim_s"] > 0].copy()
    drive["speed"] = drive["distance_m"] / drive["sim_s"]
    drive["red_frac"] = drive["red_s"] / drive["sim_s"]
    observed = drive.groupby("town")[["speed", "red_frac"]].mean()

    lines = ["| town | campaign m/s | calibration m/s | Δ | campaign red % "
             "| calibration red % |", "|---|---|---|---|---|---|"]
    for town, row in observed.iterrows():
        reference = CALIBRATION_REFERENCE.get(town)
        if reference is None:
            lines.append(
                f"| {town} | {row['speed']:.2f} | ⚠ not recorded | — "
                f"| {100 * row['red_frac']:.1f}% | — |")
            continue
        lines.append(
            f"| {town} | {row['speed']:.2f} | {reference['speed_mps']:.2f} "
            f"| {100 * (row['speed'] / reference['speed_mps'] - 1):+.1f}% "
            f"| {100 * row['red_frac']:.1f}% "
            f"| {100 * reference['red_fraction']:.1f}% |")
    display(Markdown("\n".join(lines)))
    display(Markdown(
        "A `⚠ not recorded` row means section 3's `CALIBRATION_REFERENCE` "
        "was never copied into the config cell."))

## 10. Is the service window kept?

A **breach** is a fulfilled rider whose wait exceeded the window they were
accepted under.

!!! warning "At the operating point this is zero by construction"
    This campaign runs `window == patience`, which *is* the recommendation
    being demonstrated. Anyone who waits past the window has therefore
    already been cancelled, so no fulfilled request can breach and the rate
    is a flat zero — **abandonment enforces the promise on the policy's
    behalf.** That is a result worth stating in one line, and it is not a
    measurement of policy behaviour.

    Measuring breaches needs a window *tighter* than patience. The synthetic
    reference is W750 against 900 s patience: **3.4%**, essentially all of it
    structural — the progress fallback drives to the nearest pending pickup
    ignoring the window when nothing can be scheduled. On Town01's geometry
    the same setup gives **2.4% for Reordering and 3.1% for Window**, which
    corroborates it closely. Running that arm on CARLA would cost a second
    campaign, and breach rate is not one of the four §3.1 KPIs.

So the honest sentence is: *at the recommended operating point, the window is
kept for every rider who is served at all.* The cell below verifies that
rather than charting it.

In [ ]:
if not HAVE_RUNS:
    display(Markdown("_Nothing to report until the campaign has run._"))
elif not BREACH_MEASURABLE:
    served_only = frame[frame["policy"] != "Fixed route"]
    worst = served_only["breach"].max()
    display(Markdown(
        f"`window == patience == {WINDOW:.0f} s`, so a fulfilled rider "
        f"cannot breach by construction. Observed maximum across "
        f"{len(served_only)} runs: **{worst:.2f}%** — anything but 0.00 "
        f"would mean the abandonment logic and the window disagree, which "
        f"would be a bug rather than a finding.\n\n"
        f"**Claim:** at the recommended operating point the service window "
        f"is kept for every rider who is served at all."))
    display(served_only.groupby(["town", "policy"])["breach"]
            .agg(["mean", "max"]).round(3))
else:
    breaches = frame[frame["policy"] != "Fixed route"]
    figure, axes = plt.subplots(
        1, len(TOWNS_WITH_RUNS), figsize=(4.2 * len(TOWNS_WITH_RUNS), 3.6),
        sharey=True, squeeze=False)
    for axis, town in zip(axes[0], TOWNS_WITH_RUNS):
        block = breaches[breaches["town"] == town]
        for colour, label in zip(SERIES[1:], POLICY_LABELS[1:]):
            subset = block[block["policy"] == label].sort_values("seed")
            axis.plot(subset["seed"], subset["breach"], marker="o",
                      markersize=6, lw=2.0, color=colour, label=label)
        axis.axhline(3.4, ls="--", lw=1.1, color=INK_SOFT)
        axis.set_title(town)
        axis.set_xlabel("seed")
        axis.grid(axis="y"); axis.set_axisbelow(True)
    axes[0][0].set_ylabel("% of fulfilled riders")
    axes[0][-1].legend()
    figure.tight_layout()
    plt.show()
    display(breaches.groupby(["town", "policy"])["breach"]
            .agg(["mean", "max"]).round(2))

## 11. What we concluded — Town01, 2026-08-11

30 runs, 3 policies x 10 seeds, 20 NPCs, 2.29 h at 4.6 min/trial. The first
campaign against the calibrated travel model.

### 1. §3.2.2 transfers, and the gate did real work

| policy | served | wait | ride | throughput | sum | verdict |
|---|---|---|---|---|---|---|
| Fixed route | 63.3% | 347 s | 561 s | 0.30 | — | — |
| Window W750 | **89.7%** | 355 s | 390 s | 0.46 | **+81.5%** | **PASS** |
| Reordering W750 | **87.3%** | 372 s | 367 s | 0.45 | **+76.4%** | **PASS** |

The win is **ride time (+31/+35%) and throughput (+53/+49%)**, not wait.

!!! warning "Fixed route's wait is survivorship — do not quote it as competitive"
    347 s is the mean over the **19 requests it managed**, ignoring the 11 it
    abandoned. A policy that gives up on its hard riders posts good wait and
    ride numbers by construction, which is why the fulfilment gate has to be
    read first and the sum never read alone.

Quote this as **demonstrated transfer at n=10 seeds**, never as an
independent statistical result: the 5% CI bar deliberately lives on the
synthetic map.

### 2. L1's advantage does not replicate here — under-powered, not refuted

| metric | L0 | L1 | L1 − L0 | p |
|---|---|---|---|---|
| served | 89.7% | 87.3% | −2.3 pp | 0.37 |
| wait | 355 s | 372 s | +16.6 s | 0.20 |
| ride | 390 s | 367 s | **−23.0 s** | 0.60 |

Ride moves in the predicted direction at a third of the expected magnitude
(**−70 s** on both the synthetic map and the mock). The paired sd is ~132 s,
so resolving −70 s at p<0.05 needs about **20 seeds**; the synthetic result
used 60 on a single map.

**This is what the 2026-08-10 split anticipated.** Write it as *"the point
estimate is in the predicted direction; CARLA at n=10 cannot resolve it, and
the statistical claim rests on 25 synthetic maps"*. **Never** as "confirmed
on CARLA". Ten more seeds of L0 and L1 alone (~1.5 h) would settle it.

### 3. The calibration is ~10% optimistic for policy-chosen routes

| | campaign | calibration tour |
|---|---|---|
| effective speed | 3.77 m/s | 4.21 m/s (**−10.3%**) |
| red-light share | 30.9% | 32.1% |

Red lights transfer; overall speed does not quite. The calibration drives a
fixed tour of all fifteen stops while the policy picks a different and
slower mix of legs — and 3.77 m/s is exactly what the pre-calibration store
showed for 20-NPC runs, so this is a stable property of policy-driven routes
rather than noise. **Re-derive `traffic_efficiency` from a policy-driven run
before applying the recipe to further towns**, or carry the bias knowingly.

### 4. The integration is sound, and the window is kept

**0 repaths and 0 unroutable legs** across ~1,233 driven legs, with 4.4% of
time queued behind traffic. Breach was 0.00% everywhere — by construction at
`window == patience`, so the claim is *the window is kept for every rider who
is served at all*, not a measurement of policy behaviour.

### 5. The method worth reusing

The scenario had to be calibrated first: the initial attempt had fixed route
serving 100%, and **raising the arrival rate did not fix it** — patience was
the lever. That sizing was done on the **mock backend against Town01's real
calibrated geometry**, 480 trials in under a minute with no server, and it
predicted CARLA's fixed-route baseline to within **1 pp** (62.3% → 63.3%)
and the trial cost to within seconds (4.3 → 4.6 min).

**Size every future town's scenario on the mock first.** It costs a minute
and it is the difference between a campaign that measures something and
three hours of inert gate.